# 신용카드 고객 소득 구간 예측 및 계층적 분류 검토 보고서

## 보고서 개요
본 노트북은 `creditcard-churn` 데이터를 활용하여 `income_id`를 예측하는 과정을 실무 분석 보고서 형태로 재구성한 문서입니다.  
원본 코드의 흐름을 유지하되, **데이터 로드 → 구조 점검 → 타깃 분리 → 학습/평가 → 하이퍼파라미터 탐색 → 계층적 이진 분류 검토**의 단계로 세분화했습니다.

## 비즈니스 목적
- 고객의 누락/불명확한 소득 구간(`income_id`)을 예측할 수 있는 기준 모델을 마련합니다.
- 단일 다중분류 모델과 계층적 이진 분류 접근을 비교하여, 향후 결측 대체 및 고객 세분화 모델링의 방향성을 점검합니다.
- 팀원이 즉시 이어서 실험할 수 있도록 코드와 해설을 함께 정리합니다.

## 분석 포인트
- `income_id == 6`은 미상(Unknown) 값으로 간주하고, **알려진 소득 구간(1~5)**만으로 학습 데이터를 구성합니다.
- 먼저 **5개 소득 구간 다중분류 문제**로 접근하여 전체 예측 가능성을 점검합니다.
- 이후 **하위 구간(1,2) vs 상위 구간(3,4,5)** 이진 분류로 단순화하여 계층적 분류의 실효성을 확인합니다.
- 평가 지표는 정확도뿐 아니라 **Recall, F1, ROC-AUC, PR-AUC, Precision**까지 확인하여 클래스 불균형 및 운영 활용성을 함께 검토합니다.


## 1. 분석 환경 설정

### 비즈니스 목적
재현 가능한 분석 환경을 먼저 고정해두면, 팀원이 동일한 컨테이너/노트북 환경에서 같은 결과를 확인하기 쉽습니다.

### 기대 효과
- 실험 결과 재현성 확보
- 라이브러리 누락 및 버전 차이로 인한 오류 예방
- 이후 셀에서 공통 설정 재사용 가능

### 분석 포인트
- 시각화/표현 옵션을 초기에 맞춰 두면 결과 해석이 쉬워집니다.
- 배포 환경에서는 라이브러리 버전 차이로 API 동작이 달라질 수 있으므로, requirements 관리가 중요합니다.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import requests

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# 왜 이 작업을 하나?
# - 노트북 실행 시 표와 그래프를 팀 공통 기준으로 보기 좋게 맞추기 위해 사용합니다.
# - 분석 결과를 검토/공유할 때 출력 형식이 통일되면 커뮤니케이션 비용이 줄어듭니다.
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# 유지보수 주의:
# - 한글 폰트는 실행 환경(OS/컨테이너)에 따라 다를 수 있으므로 기본 설정으로 둡니다.
# - 서버/배포 환경에서는 폰트 관련 옵션이 없더라도 코드가 실패하지 않게 최소 설정만 유지합니다.
sns.set_theme(style="whitegrid", context="notebook")


## 2. 데이터 조회 함수 정의

### 비즈니스 목적
분석 대상 데이터를 API 형태의 파이프라인에서 직접 조회하도록 구성하면, 로컬 파일 의존도를 낮추고 운영 데이터 흐름과 유사한 방식으로 검증할 수 있습니다.

### 기대 효과
- 데이터 적재 위치가 바뀌어도 호출 인터페이스만 유지되면 분석 코드 재사용 가능
- 학습용 전체 데이터와 X/y 분리 데이터를 같은 함수로 일관되게 관리 가능

### 분석 포인트
- API 응답 구조가 바뀌면 후속 셀이 모두 영향을 받으므로, 초기에 스키마를 명확히 확인해야 합니다.
- 운영 환경에서는 네트워크 장애나 응답 지연에 대비한 예외 처리도 필요합니다.


In [ ]:
def fetch_creditcard(X_y_split: bool = False) -> pd.DataFrame | tuple[pd.DataFrame, pd.Series]:
    # 왜 이 작업을 하나?
    # - 데이터 원천을 함수로 감싸 두면, 노트북 곳곳에서 같은 요청 로직을 반복하지 않아도 됩니다.
    # - 향후 엔드포인트 변경 시 이 함수만 수정하면 되므로 유지보수가 쉬워집니다.
    response = requests.get(
        "http://pipeline:8000/dataset/creditcard-churn",
        params={"X_y_split": X_y_split},
        timeout=30,  # 네트워크 지연 시 무한 대기를 막기 위해 timeout을 지정합니다.
    )
    response.raise_for_status()

    payload = response.json()

    if X_y_split:
        # 왜 DataFrame / Series로 다시 감싸는가?
        # - 모델링 코드가 pandas 객체를 기대하는 경우가 많아 후속 처리 호환성을 높입니다.
        X = pd.DataFrame(payload["x"], index=payload["index"])
        y = pd.Series(payload["y"], index=payload["index"], name="target")
        return X, y

    df = pd.DataFrame(payload["data"], index=payload["index"])
    return df

# 유지보수 주의:
# - API 주소(http://pipeline:8000/...)는 컨테이너 네트워크 기준입니다.
# - 로컬/서버/CI 환경에 따라 hostname이 달라질 수 있으므로 환경변수화가 권장됩니다.
# - 응답 JSON의 key 이름(data, x, y, index)이 바뀌면 즉시 오류가 발생합니다.


## 3. 원본 데이터 로드

### 비즈니스 목적
학습 전 전체 데이터의 구조를 확인하여, 어떤 컬럼이 예측에 활용 가능한지 파악합니다.

### 기대 효과
- 데이터 크기와 컬럼 수를 빠르게 파악
- 이후 전처리/모델링 범위를 명확히 설정 가능

### 분석 포인트
- 현재 데이터는 정수형 ID 컬럼과 연속형 거래/한도 변수로 구성되어 있습니다.
- `income_id`는 이번 분석의 핵심 타깃이며, 이후 Unknown 값 분리에 사용됩니다.


In [ ]:
df = fetch_creditcard(X_y_split=False)

# 왜 display()를 쓰는가?
# - Jupyter 환경에서 DataFrame을 더 깔끔하게 확인할 수 있어, 팀원이 구조를 빠르게 파악할 수 있습니다.
display(df.head())
print(f"데이터 크기: {df.shape[0]:,}행 × {df.shape[1]:,}열")


## 4. 주요 타깃 분포 사전 점검

### 비즈니스 목적
타깃(`income_id`)과 관련 주요 범주형 코드 분포를 먼저 보면, 모델 난이도와 데이터 편중 가능성을 빠르게 파악할 수 있습니다.

### 기대 효과
- 특정 클래스 쏠림 여부를 조기에 발견
- 학습/평가 전략(예: stratify, macro 평균 지표) 선정 근거 확보

### 분석 포인트
- `income_id`는 일부 구간에 데이터가 집중되어 있습니다.
- `education_id` 역시 특정 코드 비중이 높아, 소득 예측에 간접적 설명 변수가 될 가능성이 있습니다.


In [ ]:
income_counts = df["income_id"].value_counts().sort_index()
education_counts = df["education_id"].value_counts().sort_index()

display(pd.DataFrame({"income_id_count": income_counts}))
display(pd.DataFrame({"education_id_count": education_counts}))


## 5. 데이터 구조 및 결측 여부 점검

### 비즈니스 목적
모델 학습 전에 자료형과 결측 상태를 확인해야 불필요한 형변환 오류나 학습 실패를 예방할 수 있습니다.

### 기대 효과
- 수치형 모델(XGBoost)에 바로 투입 가능한지 판단
- 결측치/타입 이슈를 조기 발견하여 디버깅 시간 단축

### 분석 포인트
- 현재 컬럼은 모두 수치형으로 구성되어 있어 XGBoost 입력에 적합합니다.
- 표면적으로 결측치는 없어 보이지만, `income_id == 6`처럼 **의미상 결측**은 별도로 처리해야 합니다.


In [ ]:
# 왜 info()와 isnull()을 함께 보는가?
# - info()는 dtype/nonnull 수를 보고,
# - isnull().sum()은 컬럼별 결측 개수를 명시적으로 확인하기 위해 사용합니다.
df.info()

null_summary = df.isnull().sum().to_frame("null_count")
display(null_summary[null_summary["null_count"] > 0])

print(f"전체 결측치 수: {int(df.isnull().sum().sum())}")


## 6. Unknown 소득 구간 분리

### 비즈니스 목적
이번 분석의 목적은 **알려진 소득 구간 패턴을 학습한 뒤, Unknown 고객의 소득 구간을 추정할 수 있는 기반**을 만드는 것입니다.

### 기대 효과
- 학습 데이터와 예측 대상 데이터를 명확히 분리
- 의미상 결측(target unknown)을 학습에 섞지 않아 레이블 오염 방지

### 분석 포인트
- `income_id == 6`은 미상(Unknown)으로 해석합니다.
- 따라서 학습은 `income_id != 6` 데이터만 사용하고, `income_id == 6`은 추후 예측 적용 대상으로 남겨둡니다.


In [ ]:
known_df = df[df["income_id"] != 6].copy()
unknown_df = df[df["income_id"] == 6].copy()

summary_df = pd.DataFrame(
    {
        "dataset": ["known_df", "unknown_df"],
        "rows": [known_df.shape[0], unknown_df.shape[0]],
        "cols": [known_df.shape[1], unknown_df.shape[1]],
    }
)

display(summary_df)


## 7. 모델 입력(X)과 타깃(y) 구성

### 비즈니스 목적
예측 대상인 `income_id`를 분리하여 모델 입력과 정답 레이블을 명확하게 구분합니다.

### 기대 효과
- 데이터 누수(leakage) 방지
- 학습/평가 파이프라인 단순화

### 분석 포인트
- `income_id`는 타깃이므로 입력 변수에서 반드시 제거해야 합니다.
- ID 성격이 강한 `creditcard_churn_id`는 설명력이 낮을 가능성이 있으나, 이번 원본 코드 흐름을 유지하기 위해 우선 포함합니다.
- 실무에서는 ID 컬럼 제거 실험을 추가로 권장합니다.


In [ ]:
X = known_df.drop(columns=["income_id"])
y = known_df["income_id"]

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head().to_frame("income_id"))


## 8. 학습/평가 데이터 분리

### 비즈니스 목적
모델이 학습 데이터만 외우는 것이 아니라, 보지 못한 데이터에서도 일반화되는지 확인하기 위해 train/test split을 수행합니다.

### 기대 효과
- 과적합 여부 점검 가능
- 실제 운영 환경과 유사한 검증 구조 확보

### 분석 포인트
- `stratify=y`를 적용하여 소득 구간 비율이 train/test에 유사하게 유지되도록 합니다.
- 클래스 불균형이 존재하므로 stratify는 필수에 가깝습니다.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,  # 왜 필요한가? 클래스 비율 왜곡을 막아 평가 신뢰도를 높이기 위함입니다.
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)


## 9. 분할 결과 검증 리포트

### 비즈니스 목적
데이터를 나눈 뒤 클래스 비율이 실제로 잘 유지되었는지 확인하여, 이후 평가 지표가 왜곡되지 않도록 합니다.

### 기대 효과
- 분할 결과 신뢰성 확보
- 클래스별 비교 가능한 평가 환경 구성

### 분석 포인트
- train/test의 클래스 비율이 거의 동일하면 stratified split이 정상 작동한 것입니다.
- 특정 클래스가 test에서 지나치게 적어지면 accuracy 해석이 불안정해질 수 있습니다.


In [ ]:
def print_split_report(X_train, X_test, y_train, y_test):
    # 왜 이 함수를 만드는가?
    # - 분할 결과를 팀 공통 포맷으로 빠르게 점검하기 위함입니다.
    # - 실험을 반복할 때 split 품질을 매번 같은 기준으로 확인할 수 있습니다.
    print(f"\n{'Train / Test Split Summary':^50}")
    print("=" * 50)
    print(f"X_train shape : {X_train.shape}")
    print(f"X_test  shape : {X_test.shape}")
    print("-" * 50)
    print(f"y_train shape : {y_train.shape}")
    print(f"y_test  shape : {y_test.shape}")
    print("-" * 50)
    print("Target Distribution (Train)")
    print(y_train.value_counts().sort_index().to_string())
    print("-" * 50)
    print("Target Distribution (Test)")
    print(y_test.value_counts().sort_index().to_string())
    print("-" * 50)
    print("Train Ratio")
    print(y_train.value_counts(normalize=True).sort_index().to_string())
    print("-" * 50)
    print("Test Ratio")
    print(y_test.value_counts(normalize=True).sort_index().to_string())
    print("=" * 50)

print_split_report(X_train, X_test, y_train, y_test)


## 10. 평가 함수 정의

### 비즈니스 목적
다중분류와 이진분류를 모두 같은 형식으로 비교할 수 있도록 공통 평가 함수를 정의합니다.

### 기대 효과
- 실험 간 비교 일관성 확보
- 정확도 외 지표까지 함께 확인하여 운영 관점 해석 가능

### 분석 포인트
- 정확도만 보면 클래스 편중 상황을 놓칠 수 있으므로, Recall/F1/ROC-AUC/PR-AUC를 함께 봅니다.
- 특히 PR-AUC는 불균형 문제에서 실제 탐지 품질을 해석하는 데 유용합니다.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.multiclass import type_of_target

def evaluate_predictions(y_true, y_pred, y_proba, classes=None, average="macro", dataset_name="Dataset"):
    # 왜 공통 평가 함수를 쓰는가?
    # - 다중분류/이진분류를 같은 형식으로 기록해 실험 비교를 쉽게 하기 위함입니다.
    # - 지표 계산 로직을 한곳에 모으면 실수와 중복 코드를 줄일 수 있습니다.
    target_type = type_of_target(y_true)

    result = {
        "dataset": dataset_name,
        "target_type": target_type,
        "accuracy": accuracy_score(y_true, y_pred),
    }

    y_proba = np.asarray(y_proba)

    if target_type == "binary":
        result["recall"] = recall_score(y_true, y_pred)
        result["f1"] = f1_score(y_true, y_pred)

        # 왜 확률 shape을 점검하는가?
        # - ROC-AUC / PR-AUC는 확률 점수가 필요하며, 잘못된 shape이면 계산이 왜곡되거나 실패합니다.
        if y_proba.ndim == 2 and y_proba.shape[1] == 2:
            y_score = y_proba[:, 1]
        elif y_proba.ndim == 1:
            y_score = y_proba
        else:
            raise ValueError(f"Binary classification shape error: {y_proba.shape}")

        result["roc_auc"] = roc_auc_score(y_true, y_score)
        result["pr_auc"] = average_precision_score(y_true, y_score)
        result["precision_score"] = precision_score(y_true, y_pred)

    elif target_type == "multiclass":
        result["recall"] = recall_score(y_true, y_pred, average=average)
        result["f1"] = f1_score(y_true, y_pred, average=average)

        if y_proba.ndim != 2:
            raise ValueError(f"Multiclass classification에서는 y_proba가 2차원이어야 합니다: {y_proba.shape}")

        result["roc_auc"] = roc_auc_score(
            y_true,
            y_proba,
            multi_class="ovr",
            average=average,
        )

        if classes is None:
            classes = np.unique(y_true)

        y_true_bin = label_binarize(y_true, classes=classes)
        result["pr_auc"] = average_precision_score(
            y_true_bin,
            y_proba,
            average=average,
        )
        result["precision_score"] = precision_score(y_true, y_pred, average=average)

    else:
        raise ValueError(f"지원하지 않는 target type입니다: {target_type}")

    return result


def print_report(res):
    print("=" * 20, f"{res['dataset']}", "=" * 20)
    print(f"{'target type':>20}: {res['target_type']}")
    for k in [key for key in res if key not in ["dataset", "target_type"]]:
        print(f"{k:>20}: {res[k]:.6f}")

# 유지보수 주의:
# - 다중분류 클래스 라벨 순서(classes)는 y_proba의 컬럼 순서와 일치해야 합니다.
# - 추후 라벨 체계가 바뀌면 label_binarize 기준도 함께 수정해야 합니다.


## 11. 기준 모델 1차 학습: 다중분류 XGBoost

### 비즈니스 목적
우선 5개 소득 구간(1~5)을 한 번에 맞추는 다중분류 모델을 학습해, 문제 난이도와 기준 성능을 파악합니다.

### 기대 효과
- 소득 구간 예측의 기본 가능성 점검
- 향후 튜닝/계층적 분류 성능 비교 기준 확보

### 분석 포인트
- XGBoost는 비선형 관계와 변수 상호작용을 잘 잡는 트리 기반 모델입니다.
- 스케일링이 필수는 아니므로, 현재 수치형 입력 구조와 잘 맞습니다.
- 다만 라벨이 1~5이므로, XGBoost 입력 전 0~4로 인코딩 후 예측 결과를 다시 복원합니다.


In [ ]:
from xgboost import XGBClassifier

xgb_income = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    eval_metric="mlogloss",
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    tree_method="hist",
)

# 왜 y_train - 1을 하는가?
# - XGBoost 다중분류는 일반적으로 0부터 시작하는 클래스 인덱스를 기대하므로 라벨 체계를 맞춰줍니다.
xgb_income.fit(X_train, y_train - 1)

y_train_pred_enc = xgb_income.predict(X_train)
y_train_proba = xgb_income.predict_proba(X_train)
y_train_pred = y_train_pred_enc + 1  # 원래 소득 코드(1~5)로 복원

y_test_pred_enc = xgb_income.predict(X_test)
y_test_proba = xgb_income.predict_proba(X_test)
y_test_pred = y_test_pred_enc + 1


## 12. 기준 모델 1차 성능 평가

### 비즈니스 목적
기준 모델이 실제로 어느 정도 일반화되는지 train/test 성능 차이를 통해 확인합니다.

### 기대 효과
- 과적합 여부 조기 파악
- 이후 튜닝 필요성 판단

### 분석 포인트
- Train 성능 대비 Test 성능이 낮으면 과적합 가능성을 의심할 수 있습니다.
- Accuracy뿐 아니라 macro F1, macro Recall도 함께 확인해야 클래스별 고른 성능 여부를 해석할 수 있습니다.


In [ ]:
income_train_result = evaluate_predictions(
    y_true=y_train,
    y_pred=y_train_pred,
    y_proba=y_train_proba,
    classes=[1, 2, 3, 4, 5],
    average="macro",
    dataset_name="Income Train XGBoost (Baseline)",
)

income_test_result = evaluate_predictions(
    y_true=y_test,
    y_pred=y_test_pred,
    y_proba=y_test_proba,
    classes=[1, 2, 3, 4, 5],
    average="macro",
    dataset_name="Income Test XGBoost (Baseline)",
)

print_report(income_train_result)
print_report(income_test_result)


## 13. 예측 결과 분포 확인

### 비즈니스 목적
모델이 특정 소득 구간만 과도하게 예측하는지 확인하여, 실제 운영 적용 가능성을 점검합니다.

### 기대 효과
- 클래스 쏠림 여부 파악
- 소수 클래스 예측 실패 패턴 탐색

### 분석 포인트
- 실제 분포와 예측 분포 차이가 크면, 특정 클래스 편향이나 경계 학습 실패를 의심할 수 있습니다.


In [ ]:
pred_dist_df = pd.DataFrame({
    "actual_test": y_test.value_counts().sort_index(),
    "pred_test": pd.Series(y_test_pred).value_counts().sort_index(),
}).fillna(0).astype(int)

display(pred_dist_df)


## 14. 간단 수치 시각화: 실제 분포 vs 예측 분포

### 비즈니스 목적
텍스트 표만으로는 클래스 편향이 직관적으로 보이지 않을 수 있으므로, 분포를 시각적으로 비교합니다.

### 기대 효과
- 소득 구간별 과소/과대 예측을 빠르게 식별
- 튜닝 방향 설정 보조

### 분석 포인트
- 실제 분포와 예측 분포 막대 차이가 큰 구간은 오분류 위험이 높은 클래스일 가능성이 있습니다.


In [ ]:
plot_df = pred_dist_df.reset_index().rename(columns={"index": "income_id"})
plot_df = plot_df.melt(id_vars="income_id", value_vars=["actual_test", "pred_test"],
                       var_name="type", value_name="count")

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="income_id", y="count", hue="type")
plt.title("테스트셋 실제 소득 구간 분포 vs 예측 분포")
plt.xlabel("income_id")
plt.ylabel("건수")
plt.legend(title="구분")
plt.tight_layout()
plt.show()


## 15. 수동 하이퍼파라미터 탐색 함수 정의

### 비즈니스 목적
핵심 파라미터 조합을 빠르게 실험하여, 성능 개선 가능성이 있는 영역을 탐색합니다.

### 기대 효과
- 전면적인 GridSearch 이전에 좁은 범위를 빠르게 확인
- 계산 자원 절감

### 분석 포인트
- `n_estimators`, `min_child_weight`, `max_depth`는 트리 복잡도와 일반화 성능에 큰 영향을 줍니다.
- 지나치게 깊은 트리는 Train 성능은 높이지만 Test 일반화 성능이 떨어질 수 있습니다.


In [ ]:
def my_grid_search(n_estim, min_child_len, max_depth):
    max_score = 0
    best_params = []

    for i in n_estim:
        for j in min_child_len:
            for k in max_depth:
                print(f"n_estimators: {i}, min_child_weight: {j}, max_depth: {k}")

                model = XGBClassifier(
                    objective="multi:softprob",
                    num_class=5,
                    eval_metric="mlogloss",
                    n_estimators=i,
                    max_depth=k,
                    learning_rate=0.05,
                    min_child_weight=j,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_lambda=1.0,
                    random_state=42,
                    tree_method="hist",
                )

                model.fit(X_train, y_train - 1)

                y_test_pred_enc = model.predict(X_test)
                y_test_pred = y_test_pred_enc + 1
                sc = accuracy_score(y_test, y_test_pred)

                if max_score < sc:
                    max_score = sc
                    best_params = [i, j, k]

    print(
        f"Best Score: {max_score:.6f} "
        f"with params: n_estimators={best_params[0]}, "
        f"min_child_weight={best_params[1]}, max_depth={best_params[2]}"
    )

# 유지보수 주의:
# - 이 함수는 validation set 재사용 기반이라, 반복 탐색 시 test leakage 성격이 생길 수 있습니다.
# - 최종 모델 선정은 교차검증 기반 결과로 확정하는 것이 더 안전합니다.


## 16. 수동 탐색 실행

### 비즈니스 목적
우선 작은 범위에서 성능 개선 조합을 확인하여, 정교한 튜닝의 출발점을 잡습니다.

### 기대 효과
- 계산량을 크게 늘리지 않고 탐색 시작점 확보
- 실무적으로 빠른 베이스라인 개선 가능

### 분석 포인트
- 여기서 확인한 최적 조합은 **탐색용 후보**로 해석해야 하며, 최종 확정은 교차검증 결과와 함께 봐야 합니다.


In [ ]:
my_grid_search([50], [3, 4, 5], [8, 9, 10, 11])


## 17. GridSearchCV 1차 탐색

### 비즈니스 목적
교차검증 기반으로 주요 파라미터 조합을 체계적으로 비교하여, 특정 split에만 유리한 결과를 줄입니다.

### 기대 효과
- 단일 hold-out 편향 감소
- 더 신뢰도 높은 파라미터 후보 확보

### 분석 포인트
- `cv=3`은 계산량과 검증 안정성의 균형을 고려한 설정입니다.
- scoring을 accuracy로 두었으므로, 필요 시 macro F1 기준 재탐색도 고려할 수 있습니다.


In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    "n_estimators": [100, 200, 300],
    "min_child_weight": [1, 3, 5],
    "max_depth": [3, 5, 7],
}

xgb_income_cv = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    eval_metric="aucpr",
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    tree_method="hist",
)

grid_search = GridSearchCV(
    estimator=xgb_income_cv,
    param_grid=params,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=2,
)

grid_search.fit(X_train, y_train - 1)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)


## 18. GridSearchCV 2차 세부 탐색

### 비즈니스 목적
1차 탐색 결과 주변의 세부 구간을 추가로 확인하여, 과도한 탐색 없이 미세 조정 효과를 확인합니다.

### 기대 효과
- 탐색 범위 축소를 통한 효율적 튜닝
- 기준 모델 대비 개선 여부 확인

### 분석 포인트
- 1차에서 유망했던 `max_depth=5` 주변에서 `n_estimators`, `min_child_weight`를 재탐색합니다.
- 성능 개선 폭이 매우 작다면, 복잡도 증가 대비 실익이 낮을 수 있습니다.


In [ ]:
params = {
    "n_estimators": [50, 100, 150],
    "min_child_weight": [4, 5, 6],
}

xgb_income_cv_refined = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    eval_metric="mlogloss",
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    max_depth=5,
    random_state=42,
    tree_method="hist",
)

grid_search_refined = GridSearchCV(
    estimator=xgb_income_cv_refined,
    param_grid=params,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=2,
)

grid_search_refined.fit(X_train, y_train - 1)

print("Best Parameters:", grid_search_refined.best_params_)
print("Best CV Score:", grid_search_refined.best_score_)


## 19. 튜닝 반영 최종 다중분류 모델 학습

### 비즈니스 목적
수동 탐색에서 확인된 조합을 실제 모델에 반영하여, 기준 모델 대비 개선 여부를 점검합니다.

### 기대 효과
- 성능 소폭 개선 여부 확인
- 후속 Unknown 예측 적용을 위한 후보 모델 확보

### 분석 포인트
- 본 코드 기준으로는 `n_estimators=50`, `min_child_weight=3`, `max_depth=9` 조합을 적용했습니다.
- 다만 이 조합은 hold-out test에 맞춘 결과일 수 있어, 운영 반영 전 재검증이 필요합니다.


In [ ]:
xgb_income_tuned = XGBClassifier(
    objective="multi:softprob",
    n_estimators=50,
    min_child_weight=3,
    num_class=5,
    eval_metric="mlogloss",
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    max_depth=9,
    random_state=42,
    tree_method="hist",
)

xgb_income_tuned.fit(X_train, y_train - 1)

y_train_pred_enc_tuned = xgb_income_tuned.predict(X_train)
y_train_proba_tuned = xgb_income_tuned.predict_proba(X_train)
y_train_pred_tuned = y_train_pred_enc_tuned + 1

y_test_pred_enc_tuned = xgb_income_tuned.predict(X_test)
y_test_proba_tuned = xgb_income_tuned.predict_proba(X_test)
y_test_pred_tuned = y_test_pred_enc_tuned + 1


## 20. 튜닝 모델 성능 평가

### 비즈니스 목적
기준 모델과 튜닝 모델의 성능 차이를 정량적으로 비교합니다.

### 기대 효과
- 튜닝 실효성 판단
- 최종 운영 후보 모델 선정 근거 확보

### 분석 포인트
- Test accuracy가 소폭 상승하더라도 macro F1/precision이 함께 개선되는지 확인해야 합니다.
- Train/Test 격차가 여전히 크면, 구조적으로 다중분류 난도가 높다는 신호일 수 있습니다.


In [ ]:
income_train_result_tuned = evaluate_predictions(
    y_true=y_train,
    y_pred=y_train_pred_tuned,
    y_proba=y_train_proba_tuned,
    classes=[1, 2, 3, 4, 5],
    average="macro",
    dataset_name="Income Train XGBoost (Tuned)",
)

income_test_result_tuned = evaluate_predictions(
    y_true=y_test,
    y_pred=y_test_pred_tuned,
    y_proba=y_test_proba_tuned,
    classes=[1, 2, 3, 4, 5],
    average="macro",
    dataset_name="Income Test XGBoost (Tuned)",
)

print_report(income_train_result_tuned)
print_report(income_test_result_tuned)


## 21. 기준 모델 vs 튜닝 모델 비교표

### 비즈니스 목적
성능 변화를 표 형태로 정리하여, 팀 의사결정에 바로 사용할 수 있게 합니다.

### 기대 효과
- 모델 선택 근거 가시화
- 발표/회의 자료로 재사용 가능

### 분석 포인트
- 단일 지표가 아니라 여러 지표를 함께 보고 판단해야 합니다.
- Accuracy 상승이 작고 다른 지표 개선이 미미하면, 추가 feature engineering이 더 효과적일 수 있습니다.


In [ ]:
compare_df = pd.DataFrame([
    income_train_result,
    income_test_result,
    income_train_result_tuned,
    income_test_result_tuned,
])

display(compare_df)


## 22. 계층적 분류 가설: 하위 소득군 vs 상위 소득군

### 비즈니스 목적
5개 클래스를 한 번에 맞추는 대신, 먼저 **하위 소득군(1,2)** 과 **상위 소득군(3,4,5)** 을 구분하면 더 안정적인 분류가 가능한지 검토합니다.

### 기대 효과
- 복잡한 다중분류를 단계적으로 단순화
- 향후 계층형 파이프라인 설계 가능성 확인

### 분석 포인트
- 이 단계는 “정확한 소득 구간”보다 “소득군 대분류”를 먼저 맞추는 전략입니다.
- 실제 서비스에서는 1단계 이진 분류 후, 각 그룹 내부에서 세부 클래스 분류를 수행하는 구조로 확장할 수 있습니다.


In [ ]:
y_train_bin = (~y_train.isin([1, 2])).astype(int)
y_test_bin = (~y_test.isin([1, 2])).astype(int)

# 0: 하위 소득군(1,2), 1: 상위 소득군(3,4,5)
print("Train unique classes:", np.unique(y_train_bin))
print("Test unique classes :", np.unique(y_test_bin))

display(pd.DataFrame({
    "train_count": y_train_bin.value_counts().sort_index(),
    "test_count": y_test_bin.value_counts().sort_index(),
}))


## 23. 계층적 1단계 이진 분류 모델 학습

### 비즈니스 목적
소득군 대분류 문제를 이진 분류로 바꾸어, 구분 가능성이 얼마나 높아지는지 확인합니다.

### 기대 효과
- 다중분류 대비 더 높은 안정적 성능 기대
- 계층적 분류 전략의 1단계 타당성 검증

### 분석 포인트
- 이진 분류는 문제 난도가 낮아져 성능이 크게 개선될 수 있습니다.
- 다만 이것만으로 최종 소득 구간 예측이 끝나는 것은 아니며, 2단계 세부 분류가 추가로 필요합니다.


In [ ]:
xgb_income_bin = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    tree_method="hist",
)

xgb_income_bin.fit(X_train, y_train_bin)


## 24. 이진 분류 예측 결과 점검

### 비즈니스 목적
이진 분류 모델이 정상적으로 예측값과 확률값을 생성하는지 먼저 확인합니다.

### 기대 효과
- 평가 지표 계산 전 shape 오류 예방
- 예측 파이프라인 안정성 확인

### 분석 포인트
- 이진 분류의 `predict_proba`는 일반적으로 `(n_samples, 2)` 형태여야 합니다.
- shape가 예상과 다르면 ROC-AUC/PR-AUC 계산 단계에서 오류가 발생할 수 있습니다.


In [ ]:
y_train_pred_bin = xgb_income_bin.predict(X_train)
y_train_proba_bin = xgb_income_bin.predict_proba(X_train)

y_test_pred_bin = xgb_income_bin.predict(X_test)
y_test_proba_bin = xgb_income_bin.predict_proba(X_test)

print("y_train_pred shape :", y_train_pred_bin.shape)
print("y_train_proba shape:", y_train_proba_bin.shape)
print("unique y_train_pred:", np.unique(y_train_pred_bin))


## 25. 계층적 1단계 이진 분류 성능 평가

### 비즈니스 목적
하위/상위 소득군 구분이 실제로 높은 성능을 보이는지 정량 확인합니다.

### 기대 효과
- 계층적 분류 전략 채택 여부 판단
- 다중분류 대비 구조적 개선 가능성 확인

### 분석 포인트
- Test 성능이 90% 내외로 높다면, 1단계 대분류는 충분히 실용적일 가능성이 있습니다.
- 이후에는
  1. `1 vs 2` 분류기  
  2. `3 vs 4 vs 5` 분류기  
  를 각각 추가해 완전한 계층형 구조로 발전시킬 수 있습니다.


In [ ]:
income_train_result_bin = evaluate_predictions(
    y_true=y_train_bin,
    y_pred=y_train_pred_bin,
    y_proba=y_train_proba_bin,
    classes=[0, 1],
    average="macro",
    dataset_name="Income Train XGBoost (Binary Group)",
)

income_test_result_bin = evaluate_predictions(
    y_true=y_test_bin,
    y_pred=y_test_pred_bin,
    y_proba=y_test_proba_bin,
    classes=[0, 1],
    average="macro",
    dataset_name="Income Test XGBoost (Binary Group)",
)

print_report(income_train_result_bin)
print_report(income_test_result_bin)


## 26. 최종 해석 및 실무 제언

### 비즈니스 목적
모델 결과를 단순 수치 나열이 아니라, 실제 다음 액션으로 연결 가능한 형태로 정리합니다.

### 핵심 결론
1. **다중분류(1~5 직접 예측)**는 기준 성능은 확보했지만, Test 성능이 상대적으로 낮아 세부 소득 구간을 한 번에 맞추는 데 한계가 있습니다.
2. 반면 **하위군(1,2) vs 상위군(3,4,5) 이진 분류**는 매우 높은 성능을 보였으므로, 계층적 분류 전략의 1단계는 충분히 유효합니다.
3. 즉, 현재 데이터 구조에서는 “한 번에 5개 분류”보다 “대분류 → 세부 분류” 접근이 더 안정적일 가능성이 큽니다.

### 실무 적용 제안
- **1단계:** 하위군 vs 상위군 이진 분류
- **2단계-A:** 하위군으로 예측된 고객에 대해 `1 vs 2` 세부 분류
- **2단계-B:** 상위군으로 예측된 고객에 대해 `3 vs 4 vs 5` 세부 분류
- 이후 Unknown(`income_id == 6`) 고객에 대해 위 파이프라인을 순차 적용

### 추가 개선 아이디어
- `creditcard_churn_id` 같은 식별자 성격 변수 제거 실험
- 클래스 불균형 대응을 위한 가중치/샘플링 검토
- confusion matrix 및 클래스별 리포트 추가
- feature importance / SHAP 기반 해석 강화
- 교차검증 기반 최종 모델 재선정

### 유지보수 및 배포 시 주의사항
- API endpoint 주소는 환경변수로 분리하는 것이 안전합니다.
- 라벨 인코딩(1~5 ↔ 0~4) 규칙은 추론 서비스와 반드시 동일해야 합니다.
- Unknown 코드값이 바뀌면(`6`이 아닌 다른 값), 분리 로직도 함께 수정해야 합니다.
- 운영 배포 시에는 학습/추론 입력 컬럼 순서가 반드시 동일해야 합니다.


In [ ]:
# 필요 시 Unknown 고객 예측을 위한 입력 데이터 예시
unknown_X = unknown_df.drop(columns=["income_id"])

print("Unknown 예측 대상 shape:", unknown_X.shape)
display(unknown_X.head())

# 왜 이 셀을 남겨두는가?
# - 현재 노트북은 모델 비교/검토 목적이지만,
# - 이후 확정된 모델로 unknown_df를 실제 예측할 때 바로 이어서 사용할 수 있도록 기반을 남겨둡니다.
